# Fitness Inspection

Small EDA notebook for comparing the completed `simple4`, `gecko4`, and `spider4` training runs.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")

RUNS = {
    "simple": Path("output/simple4/parameters_simple_run_20260605_162003.csv"),
    "gecko": Path("output/gecko4/parameters_gecko_run_20260605_161931.csv"),
    "spider": Path("output/spider4/parameters_spider_run_20260605_162017.csv"),
}

missing = [str(path) for path in RUNS.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing CSV files:\n" + "\n".join(missing))

dfs = {
    body: pd.read_csv(path)
    for body, path in RUNS.items()
}

for body, df in dfs.items():
    df["body"] = body
    df["num_of_generation"] = df["num_of_generation"].astype(int)

combined = pd.concat(dfs.values(), ignore_index=True)
combined.head()

## Quick Summary

In [ ]:
summary_rows = []
for body, df in dfs.items():
    final = df.iloc[-1]
    best_idx = df["best_fitness"].idxmax()
    best = df.loc[best_idx]
    summary_rows.append(
        {
            "body": body,
            "generations": int(df["num_of_generation"].max()),
            "final_best_fitness": final["best_fitness"],
            "final_worst_fitness": final["worst_fitness"],
            "final_best_ever_fitness": final["best_ever_fitness"],
            "peak_current_best_fitness": best["best_fitness"],
            "peak_current_best_generation": int(best["num_of_generation"]),
        }
    )

summary = pd.DataFrame(summary_rows).set_index("body")
summary

## Current Best Fitness

This is the best individual inside each generation, not necessarily the best ever retained across the whole run.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for body, df in dfs.items():
    ax.plot(
        df["num_of_generation"],
        df["best_fitness"],
        marker="o",
        markersize=3,
        linewidth=1.8,
        label=body,
    )

ax.set_title("Current Best Fitness by Generation")
ax.set_xlabel("Generation")
ax.set_ylabel("Normalized fitness")
ax.set_ylim(0, 1)
ax.legend(title="Body")
plt.show()

## Current Worst Fitness

This shows whether the whole population is improving or whether only a few strong individuals are carrying the run.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for body, df in dfs.items():
    ax.plot(
        df["num_of_generation"],
        df["worst_fitness"],
        marker="o",
        markersize=3,
        linewidth=1.8,
        label=body,
    )

ax.set_title("Current Worst Fitness by Generation")
ax.set_xlabel("Generation")
ax.set_ylabel("Normalized fitness")
ax.set_ylim(0, 1)
ax.legend(title="Body")
plt.show()

## Best-Ever Fitness

This is the running best retained by the EA. In runs without elitism, this can differ from the current generation's best.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for body, df in dfs.items():
    ax.plot(
        df["num_of_generation"],
        df["best_ever_fitness"],
        marker="o",
        markersize=3,
        linewidth=1.8,
        label=body,
    )

ax.set_title("Best-Ever Fitness by Generation")
ax.set_xlabel("Generation")
ax.set_ylabel("Normalized fitness")
ax.set_ylim(0, 1)
ax.legend(title="Body")
plt.show()

## Parent vs Offspring Best

Useful for checking whether new children are actually beating the previous generation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, (body, df) in zip(axes, dfs.items()):
    ax.plot(
        df["num_of_generation"],
        df["best_parent_fitness"],
        linewidth=1.8,
        label="parent best",
    )
    ax.plot(
        df["num_of_generation"],
        df["best_offspring_fitness"],
        linewidth=1.8,
        label="offspring best",
    )
    ax.set_title(body)
    ax.set_xlabel("Generation")
    ax.set_ylim(0, 1)
    ax.legend()

axes[0].set_ylabel("Normalized fitness")
fig.suptitle("Best Parent vs Best Offspring Fitness")
plt.tight_layout()
plt.show()